In [7]:
import pysam
import numpy as np
import pandas as pd
import glob
import os
from denovonet.variants import SingleVariant, TrioVariant, SingleVariantLRS, TrioVariantLRS
from sklearn.model_selection import train_test_split
import multiprocessing as mp
from functools import partial

# function to access a specific .bam file based on a dna-id
def get_revio_bam_path(dna_id):
    files_lst = (

        glob.glob(f'/ifs/data/research/revio/work/{dna_id}/GRCh38*/{dna_id}*.bam') + 

        glob.glob(f'/ifs/data/research/revio/work/{dna_id}/GRCh38*/BAMs_*/{dna_id}*.bam')

    )


    if len(files_lst) == 0: raise FileNotFoundError(f"No bam files found for {dna_id}")

    if len(files_lst) > 1: files_lst = sorted(files_lst, key=os.path.getmtime)



    return files_lst[-1]

In [34]:
def infer_variant_type(ref: str, alt: str) -> str:

    if len(ref) == 1 and len(alt) == 1: return "snp"

    if len(ref) == 1 and len(alt) > 1: return "ins"

    if len(ref) > 1 and len(alt) == 1:  return "del"

    return "complex"



In [38]:
dataset = pd.read_csv('/ifs/data/research/projects/gelana/denovocnn_lrs/data/revio_dnm/training_dataset.tsv', sep='\t')
dataset['variant_type'] = dataset[['ref', 'alt']].apply(lambda x: infer_variant_type(x.ref, x.alt), axis=1)
dataset = dataset[dataset['DNM_status']!='UNKNOWN']
dataset

,chrom,pos,ref,alt,hap_counts,variant_type_x,child,number_of_haplotypes,hap1_var_reads,hap1_total_reads,...,mother_number_of_haplotypes,mother_hap1_var_reads,mother_hap1_total_reads,mother_hap2_var_reads,mother_hap2_total_reads,mother_hap_na_var_reads,mother_hap_na_total_reads,mother_hap1_vaf,mother_hap2_vaf,DNM_status
0,chr1,13252342,C,T,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
1,chr1,13253645,G,C,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
2,chr1,22578139,T,A,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
3,chr1,80330738,A,G,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
4,chr1,101553191,A,T,NaN,NaN,DNA0121,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POSSIBLY_NOT_DNM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36495,chr8,34269352,T,C,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,23.0,...,2.0,0.0,17.0,0.0,15.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
36496,chr8,101557255,C,T,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,0.0,14.0,...,2.0,0.0,17.0,0.0,17.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
36497,chr8,111997555,T,C,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,13.0,13.0,...,2.0,0.0,19.0,0.0,11.0,0.0,0.0,0.0,0.0,POSSIBLY_PHASED_DNM
36498,chr9,89479695,G,C,defaultdict(<function count_variants_by_haplot...,snp,DNA24-12831,2.0,7.0,7.0,...,3.0,0.0,4.0,0.0,1.0,0.0,10.0,0.0,0.0,POSSIBLY_PHASED_DNM


In [40]:
dataset['DNM_status'].value_counts()

POSSIBLY_NOT_DNM       20500
POSSIBLY_PHASED_DNM    13014
Name: DNM_status, dtype: int64

In [3]:
# gets the id-s of all children
children = dataset['child'].values.tolist()
ids = np.unique(children)
ids

array(['DNA0121', 'DNA02-00077', 'DNA07-06065', 'DNA08-03179',
       'DNA09-06071', 'DNA09-15159', 'DNA10-12564', 'DNA11-04822',
       'DNA11-26362', 'DNA11-26445', 'DNA12-01403', 'DNA13-01861',
       'DNA13-05777', 'DNA13-05977', 'DNA13-08435', 'DNA13-11099',
       'DNA13-11484', 'DNA13-11628', 'DNA13-15626', 'DNA13-15765',
       'DNA13-16322', 'DNA14-00479', 'DNA14-19007', 'DNA14-19013',
       'DNA14-19814', 'DNA14-20628', 'DNA14-22376', 'DNA14-25790',
       'DNA14-26236', 'DNA14-32025', 'DNA14-33838', 'DNA14-36047',
       'DNA15-01061', 'DNA15-02690', 'DNA15-03061', 'DNA15-03937',
       'DNA15-06715', 'DNA15-09437', 'DNA15-11716', 'DNA15-12332',
       'DNA15-12880', 'DNA15-14076', 'DNA15-15161', 'DNA15-19905',
       'DNA16-07145', 'DNA16-09298', 'DNA16-10084', 'DNA16-11854',
       'DNA16-12636', 'DNA16-13779', 'DNA16-18715', 'DNA16-20326',
       'DNA16-22527', 'DNA17-00546', 'DNA17-02328', 'DNA17-05711',
       'DNA17-07626', 'DNA17-07798', 'DNA17-09105', 'DNA17-10057',

In [41]:
# split the data into train, validation and test
train_ids, remain = train_test_split(ids[:5], test_size=0.3, random_state=42)# 70% train
val_ids, test_ids = train_test_split(remain, test_size=0.5, random_state=42)# 15% validation and 15% test

snp_train = dataset[(dataset['child'].isin(train_ids)) & (dataset['variant_type'] == 'snp')]
snp_val = dataset[(dataset['child'].isin(val_ids)) & (dataset['variant_type'] == 'snp')]
snp_test = dataset[(dataset['child'].isin(test_ids)) & (dataset['variant_type'] == 'snp')]

ins_train = dataset[(dataset['child'].isin(train_ids)) & (dataset['variant_type'] == 'ins')]
ins_val = dataset[(dataset['child'].isin(val_ids)) & (dataset['variant_type'] == 'ins')]
ins_test = dataset[(dataset['child'].isin(test_ids)) & (dataset['variant_type'] == 'ins')]

del_train = dataset[(dataset['child'].isin(train_ids)) & (dataset['variant_type'] == 'del')]
del_val = dataset[(dataset['child'].isin(val_ids)) & (dataset['variant_type'] == 'del')]
del_test = dataset[(dataset['child'].isin(test_ids)) & (dataset['variant_type'] == 'del')]


# create a summary table of the dataset splits
columns = {'train': [len(snp_train), len(ins_train), len(del_train), (len(snp_train) + len(ins_train) + len(del_train))],
           'validation': [len(snp_val), len(ins_val), len(del_val), (len(snp_val) + len(ins_val) + len(del_val))],
           'test': [len(snp_test), len(ins_test), len(del_test), (len(snp_test) + len(ins_test) + len(del_test))]}
indexes = ['substitution', 'insertion', 'deletion', 'total']

split_ids = [train_ids, val_ids, test_ids]
split_folders = ["train", "val", "test"]

dataset_table = pd.DataFrame(data=columns, index=indexes)
dataset_table

,train,validation,test
substitution,402,123,119
insertion,29,11,4
deletion,57,15,23
total,488,149,146


              train  validation  test
substitution    402         123   119
insertion        29          11     4
deletion         57          15    23
total           488         149   146


In [46]:
trios = pd.read_csv("/ifs/data/research/revio/work/familyInfo.txt", sep= '\t')

def generate_images_per_trio(child_id, dataset_type, images_save_path): 
    try:
        child_path = get_revio_bam_path(trios.loc[trios['child'] == child_id, 'child'].iloc[0])
        mother_path = get_revio_bam_path(trios.loc[trios['child'] == child_id, 'father'].iloc[0])
        father_path = get_revio_bam_path(trios.loc[trios['child'] == child_id, 'mother'].iloc[0])
    except:
        return None
    
    # separate the dataset based on the type of the variant
    variant_types_folders = ['substitution', 'insertion', 'deletion']
    snp = dataset[(dataset['variant_type'] == 'snp')]
    ins = dataset[(dataset['variant_type'] == 'ins')]
    dels = dataset[(dataset['variant_type'] == 'del')]
    variant_types = [snp, ins, dels]
    
    for sub_dataset, variant_type_folder in zip(variant_types, variant_types_folders) :
        # print ("Processing", variant_type_folder, flush=True)
        # separate DNMs and unknown
        possible_DNMs = sub_dataset[(sub_dataset['DNM_status'] == "POSSIBLY_PHASED_DNM") & (sub_dataset['child'] == child_id)]
        ivs = sub_dataset[(sub_dataset['DNM_status'] == "POSSIBLY_NOT_DNM") & (sub_dataset['child'] == child_id)]
        # print(len(possible_DNMs))
        
        # print(len(ivs))
        
        # generate DNM images
        for row in range(len(possible_DNMs)):
            sample = possible_DNMs.iloc[row]
            
            child = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, child_path, reference_genome)
            father = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, father_path, reference_genome)
            mother = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, mother_path, reference_genome)
            trio = TrioVariantLRS(child, father, mother)
            img_save_path = f"{images_save_path}/{variant_type_folder}/{dataset_type}/DNMs/{child_id}_{sample['chrom']}_pos{sample['pos']}.png"
            #print (img_save_path, flush=True)
            TrioVariantLRS.save_image(img_save_path, np.flip(trio.image,2))
        
        # generate IVs images
        for row in range(len(ivs)):
            sample = ivs.iloc[row]
            
            child = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, child_path, reference_genome)
            father = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, father_path, reference_genome)
            mother = SingleVariantLRS(sample['chrom'], int(sample['pos']), int(sample['pos']) + 1, mother_path, reference_genome)

            trio = TrioVariantLRS(child, father, mother)
            img_save_path = f"{images_save_path}/{variant_type_folder}/{dataset_type}/IVs/{child_id}_{sample['chrom']}_pos{sample['pos']}.png"
            #print (img_save_path, flush=True)
            TrioVariantLRS.save_image(img_save_path, np.flip(trio.image,2))
    
    return None

In [47]:
reference_genome = pysam.FastaFile('/ifs/data/research/projects/ina/test_trio/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set.masked.fa')
images_save_path = "/ifs/data/research/projects/ina/DeNovoCNN_project/data_images"
# Generate RGB images
for ids in range(len(split_ids)):
    dataset_type = split_folders[ids]
    generate_images_per_trio_multiprocess = partial(generate_images_per_trio, dataset_type=dataset_type, images_save_path=images_save_path)
    pool = mp.Pool(3)
    
    _ = pool.map(generate_images_per_trio_multiprocess, split_ids[ids])
    
    pool.close()
    
    # for child_id in split_ids[ids]:
    #     # get paths to the trio files
    #     generate_images_per_trio_multiprocess(child_id)

/ifs/data/research/projects/ina/DeNovoCNN_project/code/DeNovoCNN/denovonet/variants.py:289: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  np.array(list(seq[base_start_position:base_end_position])) ==


In [48]:
import glob
import os
from collections import defaultdict
pattern = f"{images_save_path}/*/*/*/*.png"

counts = defaultdict(int)

for filepath in glob.glob(pattern):
    directory = os.path.dirname(filepath)
    counts[directory] += 1
    
for directory, count in sorted(counts.items()):

    print(f"{directory}: {count} png files")

/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/deletion/test/DNMs: 2 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/deletion/test/IVs: 21 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/deletion/train/DNMs: 11 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/deletion/train/IVs: 46 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/deletion/val/IVs: 15 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/insertion/test/IVs: 4 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/insertion/train/DNMs: 2 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/insertion/train/IVs: 27 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/insertion/val/DNMs: 6 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_images/insertion/val/IVs: 5 png files
/ifs/data/research/projects/ina/DeNovoCNN_project/data_

In [45]:
split_ids = [train_ids, val_ids, test_ids]
split_folders = ["train", "val", "test"]

dataset_table = pd.DataFrame(data=columns, index=indexes)
dataset_table

,train,validation,test
substitution,402,123,119
insertion,29,11,4
deletion,57,15,23
total,488,149,146


In [33]:
ivs = dataset[(dataset['DNM_status'] == "POSSIBLY_NOT_DNM") & (dataset['child'] == 'DNA09-06071')]
ivs["Variant type"]

KeyError: 'Variant type'

In [29]:
generate_images_per_trio('DNA09-06071', dataset_type, images_save_path)

Processing substitution


44
0


KeyboardInterrupt: 